In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "scikit-learn"]
ASSETS = [{"name": "auto-mpg.csv", "url": "/labs/2.2.2/auto-mpg.csv", "size": 17732, "sha256": "afa6a13c662fc0fb39168ad2e64ecb802b9aab7efef13639cc9937c6106b10b3"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 2.2.2 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pickle
from sklearn.ensemble import RandomForestRegressor

# 加载数据集
df = __________

# 显示前五行数据
print(__________)

# 处理缺失值
# 将 'horsepower' 列中的所有值转换为数值类型
df['horsepower'] = __________(__________, errors='coerce')
# 删除包含缺失值的行
df = __________

# 选择相关特征进行建模（定义自变量（返回一个DataFrame）和因变量）
X = __________
y = __________

# 将数据集划分为训练集和测试集（测试集占比20%）
X_train, X_test, y_train, y_test = __________(__________, random_state=42)

# 创建包含标准化和线性回归的管道
pipeline = __________([('scaler', __________),('linreg', __________)])

# 训练模型
__________

# 保存训练好的模型
with open('2.2.2_model.pkl', 'wb') as model_file:
    pickle.__________

# 预测并保存结果
y_pred = __________
results_df = pd.DataFrame(y_pred, columns=['预测结果'])
__________('2.2.2_results.txt', index=False)

# 测试模型
with open('2.2.2_report.txt', 'w') as results_file:
    results_file.write(f'训练集得分: {pipeline.score(X_train, y_train)}\n')
    results_file.write(f'测试集得分: {pipeline.score(X_test, y_test)}\n')

# 创建随机森林回归模型实例（创建的决策树的数量为100）
rf_model = __________(__________, random_state=42)
# 训练随机森林回归模型
__________

# 使用随机森林模型进行预测
y_pred_rf = __________

# 保存新的结果
results_rf_df = pd.DataFrame(y_pred_rf, columns=['预测结果'])
__________('2.2.2_results_rf.txt', index=False)

# 测试模型并保存得分
with open('2.2.2_report_rf.txt', 'w') as results_rf_file:
    results_rf_file.write(f'训练集得分: {rf_model.score(X_train, y_train)}\n')
    results_rf_file.write(f'测试集得分: {rf_model.score(X_test, y_test)}\n')
